<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-08-evaluate-the-meridian-and-orbit-agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 (graded) — Evaluate the Meridian and Orbit agents
**Course 3: AI Agents and Agentic AI with Python — Chapter 8: Evaluating agents**

**Problem brief (Dr. Ana Reyes):** "Every agent demo looks great. Then it fails 30% of the
time in the client's hands. Before anything ships, I want a real evaluation — not vibes."

**What you'll submit:** a trajectory-eval harness extended and applied to the Chapter 5
(Meridian) and Chapter 7 (Orbit) agents, both scorecards, an LLM-judge calibration against a
small human-labeled set, and one wired regression test.

## 1. A generic trajectory-eval harness

In [ ]:
class AgentTask:
    def __init__(self, name, run_fn, expected_outcome_check, expected_trajectory_check=None):
        self.name = name
        self.run_fn = run_fn
        self.expected_outcome_check = expected_outcome_check
        self.expected_trajectory_check = expected_trajectory_check

    def run(self):
        output = self.run_fn()
        outcome_ok = self.expected_outcome_check(output)
        trajectory_ok = self.expected_trajectory_check(output) if self.expected_trajectory_check else None
        return {'task': self.name, 'outcome_pass': outcome_ok, 'trajectory_pass': trajectory_ok, 'output': output}


def run_suite(tasks):
    results = [t.run() for t in tasks]
    outcome_rate = sum(r['outcome_pass'] for r in results) / len(results)
    traj_results = [r['trajectory_pass'] for r in results if r['trajectory_pass'] is not None]
    trajectory_rate = sum(traj_results) / len(traj_results) if traj_results else None
    return results, outcome_rate, trajectory_rate

## 2. Rebuild a compact version of the Chapter 5 dispute agent

In [ ]:
POLICY = {'small': 'auto-resolve', 'large': 'escalate'}

def run_dispute_agent(amount, dispute_id='TEST'):
    trajectory = ['intake', 'policy_check']
    branch = 'large' if amount >= 500 else 'small'
    if branch == 'large':
        trajectory.append('escalate')
    trajectory += ['draft_resolution', 'human_approval', 'resolve']
    outcome = 'HELD — awaiting human review' if branch == 'large' else f'refund ${amount}'
    return {'dispute_id': dispute_id, 'amount': amount, 'outcome': outcome, 'trajectory': trajectory, 'branch': branch}

dispute_tasks = [
    AgentTask('small dispute ($120) should auto-resolve',
              lambda: run_dispute_agent(120),
              lambda out: 'refund' in out['outcome'],
              lambda out: 'escalate' not in out['trajectory']),
    AgentTask('large dispute ($800) should escalate',
              lambda: run_dispute_agent(800),
              lambda out: 'HELD' in out['outcome'],
              lambda out: 'escalate' in out['trajectory']),
    AgentTask('boundary dispute ($500) should escalate (>= threshold)',
              lambda: run_dispute_agent(500),
              lambda out: 'HELD' in out['outcome'],
              lambda out: 'escalate' in out['trajectory']),
]

dispute_results, dispute_outcome_rate, dispute_traj_rate = run_suite(dispute_tasks)
print(f'Meridian dispute agent — outcome pass rate: {dispute_outcome_rate:.0%}, trajectory pass rate: {dispute_traj_rate:.0%}')
for r in dispute_results:
    print(f"  [{'PASS' if r['outcome_pass'] else 'FAIL'}] {r['task']}")

## 3. Rebuild a compact version of the Chapter 7 content pipeline

In [ ]:
def run_content_pipeline(include_caveat_in_first_draft):
    trajectory = ['research', 'write']
    draft = 'Great battery life and comfort.'
    if include_caveat_in_first_draft:
        draft += ' However, some users note occasional issues.'
    revisions = 0
    while 'however' not in draft.lower() and 'occasional' not in draft.lower() and revisions < 1:
        trajectory.append('critique')
        draft += ' Some users note occasional issues.'
        trajectory.append('revise')
        revisions += 1
    trajectory.append('edit')
    return {'output': draft, 'trajectory': trajectory, 'revisions': revisions}

content_tasks = [
    AgentTask('draft already has a caveat -> no revision needed',
              lambda: run_content_pipeline(True),
              lambda out: 'occasional' in out['output'].lower(),
              lambda out: out['revisions'] == 0),
    AgentTask('draft missing a caveat -> exactly one revision',
              lambda: run_content_pipeline(False),
              lambda out: 'occasional' in out['output'].lower(),
              lambda out: out['revisions'] == 1),
]

content_results, content_outcome_rate, content_traj_rate = run_suite(content_tasks)
print(f'Orbit content pipeline — outcome pass rate: {content_outcome_rate:.0%}, trajectory pass rate: {content_traj_rate:.0%}')

## 4. LLM-as-judge, calibrated against a small human-labeled set

In [ ]:
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

# a small human-labeled calibration set: (output text, human judgment of "is this a good blurb?")
human_labeled = [
    ('Great battery life and comfort. However, some users note occasional issues. Shop now.', True),
    ('Great battery life and comfort.', False),  # missing the caveat — a human would flag this
    ('The product exists.', False),
]

def llm_judge(text):
    if hosted_available:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
        resp = client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[{'role': 'user', 'content': f'Is this a good, honest product blurb (mentions a benefit AND a caveat)? Answer YES or NO only.\n\n{text}'}],
            max_tokens=5,
        )
        return 'yes' in resp.choices[0].message.content.lower()
    # offline heuristic judge — the same simple pattern-check as a stand-in
    t = text.lower()
    return ('battery' in t or 'comfort' in t) and any(w in t for w in ['however', 'occasional', 'some users'])

agreements = sum(llm_judge(text) == label for text, label in human_labeled)
print(f'Judge agreement with human labels: {agreements}/{len(human_labeled)} = {agreements/len(human_labeled):.0%}')
if agreements / len(human_labeled) < 0.7:
    print('WARNING: judge agreement is low — do not trust this judge\'s scores at face value yet.')

## 5. A regression test wired against a planted defect

In [ ]:
def test_dispute_escalation_threshold_regression():
    """CI-style regression test: a $500 dispute must ALWAYS escalate. If someone changes the
    threshold logic (e.g. `> 500` instead of `>= 500`), this test catches it immediately."""
    out = run_dispute_agent(500)
    assert 'escalate' in out['trajectory'], 'REGRESSION: the $500 boundary case stopped escalating!'
    print('Regression test PASSED: the $500 boundary still escalates correctly.')

test_dispute_escalation_threshold_regression()

## 6. Combined scorecard + write-up (fill in)
Summarize both agents' outcome/trajectory pass rates, the judge calibration result, and cost
(number of steps/calls per task, from the trajectories above). Which agent would you ship
as-is, and which needs more work first?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 8: Evaluating agents*